# EXP-032 | 모델별 다른 피처셋 앙상블

각 모델이 서로 다른 피처셋을 학습 → 예측 다양성 증가 → 앙상블 효과 극대화.

| 모델 | 피처셋 | 피처 수 | 근거 |
|---|---|---|---|
| LightGBM | FE-v1 | 85개 | EXP-020 최고 성능 기준 |
| CatBoost | FE-v2 + TE | 98개 | TE에 강한 CatBoost 특성 활용 |
| XGBoost  | FE-v2 + TE + ITE | 104개 | Interaction TE로 비선형 관계 포착 |

| 기준선 | EXP-020 OOF 0.74068 / 제출 0.74217 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 32
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    """FE-v1 (85개) — EXP-020 기준, LGB용"""
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

def add_derived_features_v2(df):
    """FE-v2 (89개) — v1 + 4개 추가, CAT/XGB용 베이스"""
    df = add_derived_features_v1(df)
    eps = 1e-6
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

# ── TE / ITE 설정 ──────────────────────────────────────────────────────────
TE_COLS = [
    '시술 시기 코드', '시술 유형', '특정 시술 유형',
    '배란 유도 유형', '난자 출처', '정자 출처',
    '시술 당시 나이', '총 시술 횟수', '총 임신 횟수',
]
TE_ALPHA = 10

ITE_PAIRS = [
    ('시술 당시 나이', '시술 유형'),
    ('시술 당시 나이', '특정 시술 유형'),
    ('시술 당시 나이', '난자 출처'),
    ('시술 당시 나이', '배란 유도 유형'),
    ('시술 유형',     '총 시술 횟수'),
    ('난자 출처',     '시술 유형'),
]
ITE_ALPHA = 20

# ── 피처 빌드 (TE/ITE는 fold 안에서 적용) ────────────────────────────────────
train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)

# LGB용: FE-v1
X_base_tr, X_base_te = preprocess(train_fe, test_fe)
X_lgb_train = add_derived_features_v1(X_base_tr)
X_lgb_test  = add_derived_features_v1(X_base_te)

# CAT/XGB용: FE-v2 (TE/ITE는 fold 안에서 추가)
X_v2_train = add_derived_features_v2(X_base_tr)
X_v2_test  = add_derived_features_v2(X_base_te)

# TE 적용 대상 컬럼이 실제로 존재하는지 필터
TE_COLS  = [c for c in TE_COLS  if c in X_v2_train.columns]
ITE_PAIRS = [(c1, c2) for c1, c2 in ITE_PAIRS
             if c1 in X_v2_train.columns and c2 in X_v2_train.columns]

y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

print(f'LGB  피처: {X_lgb_train.shape[1]}개')
print(f'CAT  피처: {X_v2_train.shape[1] + len(TE_COLS)}개  (v2 + {len(TE_COLS)} TE)')
print(f'XGB  피처: {X_v2_train.shape[1] + len(TE_COLS) + len(ITE_PAIRS)}개  (v2 + {len(TE_COLS)} TE + {len(ITE_PAIRS)} ITE)')

LGB  피처: 85개
CAT  피처: 98개  (v2 + 9 TE)
XGB  피처: 104개  (v2 + 9 TE + 6 ITE)


## 2. 모델 파라미터 (EXP-020 동일)

In [3]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED,
    thread_count=-1, verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998,
    depth=6, l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', random_state=SEED,
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647,
    max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595,
    colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928,
    reg_lambda=0.23932562420374562,
)
print('파라미터 설정 완료')

파라미터 설정 완료


## 3. 모델별 다른 피처셋으로 학습

In [4]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(X_lgb_train))
oof_cat = np.zeros(len(X_lgb_train))
oof_xgb = np.zeros(len(X_lgb_train))
test_lgb = np.zeros(len(X_lgb_test))
test_cat = np.zeros(len(X_lgb_test))
test_xgb = np.zeros(len(X_lgb_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgb_train, y_train), 1):
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    global_mean = y_tr.mean()

    # ── LGB: FE-v1 그대로 ─────────────────────────────────────────────────
    X_tr_lgb  = X_lgb_train.iloc[tr_idx]
    X_val_lgb = X_lgb_train.iloc[val_idx]
    X_te_lgb  = X_lgb_test

    ds_tr  = lgb.Dataset(X_tr_lgb, label=y_tr)
    ds_val = lgb.Dataset(X_val_lgb, label=y_val, reference=ds_tr)
    m = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m.predict(X_val_lgb)
    test_lgb        += m.predict(X_te_lgb) / N_FOLDS

    # ── CAT: FE-v2 + TE ──────────────────────────────────────────────────
    X_tr_cat  = X_v2_train.iloc[tr_idx].copy()
    X_val_cat = X_v2_train.iloc[val_idx].copy()
    X_te_cat  = X_v2_test.copy()
    for col in TE_COLS:
        grp = y_tr.groupby(X_tr_cat[col])
        smoothed = (grp.count() * grp.mean() + TE_ALPHA * global_mean) / (grp.count() + TE_ALPHA)
        te_col = f'{col}_te'
        X_tr_cat[te_col]  = X_tr_cat[col].map(smoothed).fillna(global_mean)
        X_val_cat[te_col] = X_val_cat[col].map(smoothed).fillna(global_mean)
        X_te_cat[te_col]  = X_te_cat[col].map(smoothed).fillna(global_mean)

    m = CatBoostClassifier(**CAT_PARAMS)
    m.fit(X_tr_cat, y_tr, eval_set=Pool(X_val_cat, y_val), use_best_model=True)
    oof_cat[val_idx] = m.predict_proba(X_val_cat)[:, 1]
    test_cat        += m.predict_proba(X_te_cat)[:, 1] / N_FOLDS

    # ── XGB: FE-v2 + TE + ITE ────────────────────────────────────────────
    X_tr_xgb  = X_tr_cat.copy()
    X_val_xgb = X_val_cat.copy()
    X_te_xgb  = X_te_cat.copy()
    for col1, col2 in ITE_PAIRS:
        key_tr  = X_tr_xgb[col1].astype(str)  + '_' + X_tr_xgb[col2].astype(str)
        key_val = X_val_xgb[col1].astype(str) + '_' + X_val_xgb[col2].astype(str)
        key_te  = X_te_xgb[col1].astype(str)  + '_' + X_te_xgb[col2].astype(str)
        grp = y_tr.groupby(key_tr)
        smoothed = (grp.count() * grp.mean() + ITE_ALPHA * global_mean) / (grp.count() + ITE_ALPHA)
        ite_col = f'{col1}_{col2}_ite'
        X_tr_xgb[ite_col]  = key_tr.map(smoothed).fillna(global_mean)
        X_val_xgb[ite_col] = key_val.map(smoothed).fillna(global_mean)
        X_te_xgb[ite_col]  = key_te.map(smoothed).fillna(global_mean)

    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X_val_xgb)[:, 1]
    test_xgb        += m.predict_proba(X_te_xgb)[:, 1] / N_FOLDS

    fold_aucs = [
        roc_auc_score(y_val, oof_lgb[val_idx]),
        roc_auc_score(y_val, oof_cat[val_idx]),
        roc_auc_score(y_val, oof_xgb[val_idx]),
    ]
    print(f'  Fold {fold}  LGB={fold_aucs[0]:.5f}  CAT={fold_aucs[1]:.5f}  XGB={fold_aucs[2]:.5f}')

auc_lgb = roc_auc_score(y_train, oof_lgb)
auc_cat = roc_auc_score(y_train, oof_cat)
auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'\n[최종] LGB(v1)={auc_lgb:.5f}  CAT(v2+TE)={auc_cat:.5f}  XGB(v2+TE+ITE)={auc_xgb:.5f}')

  Fold 1  LGB=0.73812  CAT=0.73827  XGB=0.73833
  Fold 2  LGB=0.74318  CAT=0.74338  XGB=0.74301
  Fold 3  LGB=0.74072  CAT=0.74020  XGB=0.74036
  Fold 4  LGB=0.73880  CAT=0.73864  XGB=0.73893
  Fold 5  LGB=0.74160  CAT=0.74062  XGB=0.74156

[최종] LGB(v1)=0.74047  CAT(v2+TE)=0.74022  XGB(v2+TE+ITE)=0.74043


## 4. 앙상블 비교

In [5]:
oofs  = np.stack([oof_lgb,  oof_cat,  oof_xgb],  axis=1)
tests = np.stack([test_lgb, test_cat, test_xgb], axis=1)
aucs  = np.array([auc_lgb, auc_cat, auc_xgb])

results = {}

results['Simple Average'] = (roc_auc_score(y_train, oofs.mean(axis=1)),
                              tests.mean(axis=1))
w_auc = aucs / aucs.sum()
results['AUC-weighted'] = (roc_auc_score(y_train, (oofs * w_auc).sum(axis=1)),
                            (tests * w_auc).sum(axis=1))
oof_ranks  = np.stack([rankdata(oofs[:, i])  for i in range(3)], axis=1)
test_ranks = np.stack([rankdata(tests[:, i]) for i in range(3)], axis=1)
results['Rank Average'] = (roc_auc_score(y_train, oof_ranks.mean(axis=1)),
                            test_ranks.mean(axis=1))

def optuna_objective(trial):
    w = np.array([
        trial.suggest_float('w_lgb', 0.0, 1.0),
        trial.suggest_float('w_cat', 0.0, 1.0),
        trial.suggest_float('w_xgb', 0.0, 1.0),
    ])
    w = w / w.sum()
    return roc_auc_score(y_train, (oofs * w).sum(axis=1))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(optuna_objective, n_trials=300, show_progress_bar=False)
best_w = np.array([study.best_params['w_lgb'],
                   study.best_params['w_cat'],
                   study.best_params['w_xgb']])
best_w = best_w / best_w.sum()
results['Optuna Weights'] = (roc_auc_score(y_train, (oofs * best_w).sum(axis=1)),
                              (tests * best_w).sum(axis=1))

BASELINE = 0.74068
print('=' * 65)
print(f'  LGB(v1)={auc_lgb:.5f}  CAT(v2+TE)={auc_cat:.5f}  XGB(v2+TE+ITE)={auc_xgb:.5f}')
print(f'  EXP-020 기준선: {BASELINE}')
print('-' * 65)
best_method, best_auc, best_test = '', 0, None
for method, (auc, test_pred) in results.items():
    flag = ' ←' if auc > BASELINE else ''
    print(f'  {method:20s}: {auc:.5f}  ({auc-BASELINE:+.5f} vs EXP-020){flag}')
    if auc > best_auc:
        best_auc, best_method, best_test = auc, method, test_pred
print('=' * 65)
print(f'\n최적: {best_method}  AUC={best_auc:.5f}')
print(f'Optuna 가중치  LGB={best_w[0]:.3f}  CAT={best_w[1]:.3f}  XGB={best_w[2]:.3f}')

  LGB(v1)=0.74047  CAT(v2+TE)=0.74022  XGB(v2+TE+ITE)=0.74043
  EXP-020 기준선: 0.74068
-----------------------------------------------------------------
  Simple Average      : 0.74070  (+0.00002 vs EXP-020) ←
  AUC-weighted        : 0.74070  (+0.00002 vs EXP-020) ←
  Rank Average        : 0.74071  (+0.00003 vs EXP-020) ←
  Optuna Weights      : 0.74071  (+0.00003 vs EXP-020) ←

최적: Rank Average  AUC=0.74071
Optuna 가중치  LGB=0.417  CAT=0.275  XGB=0.308


## 5. Submission 저장 & 실험 기록

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

if best_test.max() > 1.0:
    best_test = (best_test - best_test.min()) / (best_test.max() - best_test.min())
    print(f'[정규화 적용] {best_method} → 0~1 변환 완료')

sub['probability'] = best_test
auc_str   = f'{best_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}  ({best_method})')
print(f'probability 범위: [{sub["probability"].min():.4f}, {sub["probability"].max():.4f}]')

[정규화 적용] Rank Average → 0~1 변환 완료
저장: ../data/submissions/submission_exp032_조여진_07407.csv  (Rank Average)
probability 범위: [0.0000, 1.0000]


In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

if best_method == 'Optuna Weights':
    best_oof_pred = (oofs * best_w).sum(axis=1)
elif best_method == 'Rank Average':
    best_oof_pred = oof_ranks.mean(axis=1)
elif best_method == 'AUC-weighted':
    best_oof_pred = (oofs * w_auc).sum(axis=1)
else:
    best_oof_pred = oofs.mean(axis=1)
oof_binary = (best_oof_pred >= 0.5).astype(int)

n_cat_feat = X_v2_train.shape[1] + len(TE_COLS)
n_xgb_feat = n_cat_feat + len(ITE_PAIRS)
params_str = (f'LGB(FE-v1,{X_lgb_train.shape[1]}f)+CAT(FE-v2+TE,{n_cat_feat}f)+XGB(FE-v2+TE+ITE,{n_xgb_feat}f) | '
              f'best={best_method} | LGB={best_w[0]:.3f} CAT={best_w[1]:.3f} XGB={best_w[2]:.3f}')
NOTES    = '모델별 다른 피처셋: LGB=FE-v1, CAT=FE-v2+TE, XGB=FE-v2+TE+ITE. 예측 다양성 확보 목적'
INSIGHTS = (f'EXP-020 대비 {best_auc - BASELINE:+.5f} | '
            f'LGB(v1)={auc_lgb:.5f} CAT(v2+TE)={auc_cat:.5f} XGB(v2+TE+ITE)={auc_xgb:.5f}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Ensemble(LGB-v1+CAT-TE+XGB-ITE)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'mixed(v1/v2/TE/ITE)', X_lgb_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/28_mixed_fe_ensemble_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-032 기록 완료 (row 29)
